# Human Activity Recognition from Smartphone Accelerometer Data

Classifying physical activities (walking, jogging, sitting, etc.) from raw tri-axial
accelerometer signals, using engineered time- and frequency-domain features and a
gradient-boosted tree model (XGBoost).

**Authors:** Anton Robb and Ken Kahuthu — MSc Data Science, Liverpool John Moores University.

## Approach in brief

Raw accelerometer data is noisy and orientation-dependent: the same activity looks
different depending on how the phone is held. Rather than feed raw signals to a model,
we summarise each short signal *snippet* into a fixed set of **140 features** that capture
its shape, rhythm, and cross-axis structure, then train XGBoost on those features.

The validation strategy is the most important design choice here. Because the goal is to
recognise activities for **people the model has never seen**, we validate with
`GroupKFold` grouped by user — every user appears in either training or validation within
a fold, never both. This gives an honest estimate of unseen-user performance and prevents
the model from simply memorising individuals.


In [ ]:
# Core data, modelling, and signal-processing libraries
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from scipy.stats import skew, kurtosis, entropy, median_abs_deviation
from scipy.fft import rfft
import warnings

warnings.filterwarnings('ignore')

## 1. Load the data

The dataset is split across several CSVs:

- `signals*.csv` — raw tri-axial accelerometer readings (`x-axis`, `y-axis`, `z-axis`,
  `timestamp`), one row per sample, identified by `user_snippet`.
- `metadata*.csv` — the activity label for each `user_snippet`.
- `*_kaggle.csv` — the held-out snippets we must predict for the competition.

The training signals and metadata arrive in two parts each, which we concatenate into a
single training pool.


In [ ]:
PATH = './'

# Activity labels for each snippet
meta_train  = pd.read_csv(PATH + 'metadata.csv')
meta_extra  = pd.read_csv(PATH + 'metadata_test.csv')
meta_kaggle = pd.read_csv(PATH + 'metadata_kaggle.csv')

# Raw accelerometer signals (training set arrives in two files)
signals_train = pd.concat([
    pd.read_csv(PATH + 'signals.csv'),
    pd.read_csv(PATH + 'signals_test.csv')
], ignore_index=True)

signals_kaggle = pd.read_csv(PATH + 'signals_kaggle.csv')

# Full training cohort metadata
full_meta = pd.concat([meta_train, meta_extra], ignore_index=True)

print(f"Total training snippets:        {len(full_meta)}")
print(f"Snippets to predict (Kaggle):   {len(meta_kaggle)}")

## 2. Feature engineering: 140 features per snippet

This is the core of the approach. For each snippet we take the three raw axes and derive
a richer set of signals, then summarise each with statistical and spectral features.

**Derived signals (8 in total):**

- The three raw axes `x`, `y`, `z`.
- **Magnitude** `mag = sqrt(x² + y² + z²)` — the overall acceleration regardless of phone
  orientation. This is *orientation-invariant*, which directly addresses the problem that
  the same activity produces different raw-axis readings depending on how the device is held.
- **Jerk** (`jx, jy, jz, jmag`) — the first difference of each signal, i.e. how quickly
  acceleration is *changing*. Jerk separates abrupt, spiky motion (jogging) from smooth or
  static motion (sitting).

**Per-signal features (~17 each):**

- *Time domain* — mean, standard deviation, max, min, range, median, median absolute
  deviation, interquartile range, skewness, kurtosis, RMS, and zero-crossing rate. These
  describe the distribution and shape of the signal.
- *Frequency domain* (via the real FFT) — peak frequency, maximum spectral power, total
  spectral energy, spectral entropy, and spectral centroid. These capture the *rhythm* of
  the motion: periodic activities like walking and jogging have characteristic dominant
  frequencies that static activities lack.

**Cross-axis features (4):** pairwise correlations between axes plus signal magnitude area
(SMA), capturing posture and how the axes move together.

8 signals × 17 features + 4 cross-axis ≈ **140 features** per snippet.


In [ ]:
def generate_140_features(group):
    """Summarise one snippet's raw accelerometer signal into ~140 features."""
    group = group.sort_values('timestamp')

    x = group['x-axis'].values
    y = group['y-axis'].values
    z = group['z-axis'].values
    mag = np.sqrt(x**2 + y**2 + z**2)              # orientation-invariant magnitude

    # Jerk: first derivative of each signal (rate of change of acceleration)
    jx = np.diff(x, prepend=x[0])
    jy = np.diff(y, prepend=y[0])
    jz = np.diff(z, prepend=z[0])
    jmag = np.diff(mag, prepend=mag[0])

    signals = {'x': x, 'y': y, 'z': z, 'mag': mag,
               'jx': jx, 'jy': jy, 'jz': jz, 'jmag': jmag}
    feats = {}

    for name, sig in signals.items():
        # --- Time-domain statistics ---
        feats[f'{name}_mean']   = np.mean(sig)
        feats[f'{name}_std']    = np.std(sig)
        feats[f'{name}_max']    = np.max(sig)
        feats[f'{name}_min']    = np.min(sig)
        feats[f'{name}_range']  = np.ptp(sig)
        feats[f'{name}_median'] = np.median(sig)
        feats[f'{name}_mad']    = median_abs_deviation(sig, scale=1/1.4826)
        feats[f'{name}_iqr']    = np.percentile(sig, 75) - np.percentile(sig, 25)
        feats[f'{name}_skew']   = skew(sig, nan_policy='omit')
        feats[f'{name}_kurt']   = kurtosis(sig, nan_policy='omit')
        feats[f'{name}_rms']    = np.sqrt(np.mean(sig**2))

        # Zero-crossing rate: how often the (centred) signal changes sign
        sig_centered = sig - np.mean(sig)
        feats[f'{name}_zcr'] = ((sig_centered[:-1] * sig_centered[1:]) < 0).sum() / len(sig)

        # --- Frequency-domain (FFT) features ---
        fft_vals = np.abs(rfft(sig))
        psd = fft_vals**2                          # power spectral density
        psd_norm = psd / (np.sum(psd) + 1e-9)      # normalised, for entropy/centroid

        feats[f'{name}_peak_freq']     = np.argmax(fft_vals[1:]) + 1   # skip DC component
        feats[f'{name}_max_power']     = np.max(psd[1:]) if len(psd) > 1 else 0
        feats[f'{name}_spec_energy']   = np.sum(psd)
        feats[f'{name}_spec_entropy']  = entropy(psd_norm)
        feats[f'{name}_spec_centroid'] = np.sum(np.arange(len(psd)) * psd_norm)

    # --- Cross-axis posture features ---
    feats['corr_xy'] = np.corrcoef(x, y)[0, 1] if np.std(x) > 0 and np.std(y) > 0 else 0
    feats['corr_xz'] = np.corrcoef(x, z)[0, 1] if np.std(x) > 0 and np.std(z) > 0 else 0
    feats['corr_yz'] = np.corrcoef(y, z)[0, 1] if np.std(y) > 0 and np.std(z) > 0 else 0
    feats['sma']     = np.mean(np.abs(x) + np.abs(y) + np.abs(z))   # signal magnitude area

    return pd.Series(feats)

## 3. Build the feature matrices

Apply the feature generator to every snippet in the training and Kaggle sets, then attach
the activity labels. We extract `user_id` from each `user_snippet` so we can group by user
during cross-validation, and label-encode the activity strings into integer targets.


In [ ]:
# Generate features for every snippet
X_train_raw = (signals_train.groupby('user_snippet')
               .apply(generate_140_features).reset_index().fillna(0))
X_test_raw  = (signals_kaggle.groupby('user_snippet')
               .apply(generate_140_features).reset_index().fillna(0))

# Attach activity labels to the training features
train_df = full_meta.merge(X_train_raw, on='user_snippet', how='inner')

# User id (everything before the first underscore) — used to group CV folds by person
train_df['user_id'] = train_df['user_snippet'].apply(lambda s: s.split('_')[0])

feature_cols = [c for c in X_train_raw.columns if c != 'user_snippet']
X      = train_df[feature_cols]
y_str  = train_df['activity']
groups = train_df['user_id']

# Align the Kaggle test matrix to the same feature columns
X_test = (meta_kaggle[['user_snippet']]
          .merge(X_test_raw, on='user_snippet', how='left')
          .drop(columns=['user_snippet']).fillna(0))

# Encode activity labels as integers
le = LabelEncoder()
y = le.fit_transform(y_str)

print(f"Feature matrix: {X.shape[0]} snippets x {X.shape[1]} features")
print(f"Activities:     {list(le.classes_)}")

## 4. Train and validate with user-grouped cross-validation

We use `GroupKFold` grouped by `user_id`, so each fold validates on users entirely held
out of training — an honest estimate of how the model performs on **new people**.

The XGBoost hyperparameters are deliberately conservative to resist overfitting on a
modest number of users: a low learning rate with many trees and early stopping, shallow
trees (`max_depth=6`), row/column subsampling, and L1/L2 regularisation. Early stopping
halts each fold once validation log-loss stops improving.


In [ ]:
n_splits = 9
gkf = GroupKFold(n_splits=n_splits)

oof_preds  = np.zeros((len(X), len(le.classes_)))        # out-of-fold validation preds
test_probs = np.zeros((len(X_test), len(le.classes_)))   # averaged Kaggle predictions

xgb_params = {
    'objective':           'multi:softprob',
    'eval_metric':         'mlogloss',
    'tree_method':         'hist',
    'learning_rate':       0.02,
    'max_depth':           6,
    'subsample':           0.75,
    'colsample_bytree':    0.75,
    'reg_alpha':           1.5,
    'reg_lambda':          3.0,
    'min_child_weight':    4,
    'random_state':        42,
    'n_estimators':        2500,
    'early_stopping_rounds': 75,
}

fold_accs = []
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X.iloc[val_idx], y[val_idx]

    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    val_probs = model.predict_proba(X_va)
    oof_preds[val_idx] = val_probs

    acc = accuracy_score(y_va, val_probs.argmax(axis=1))
    fold_accs.append(acc)
    print(f"Fold {fold + 1} unseen-user accuracy: {acc:.4f} "
          f"(best iteration {model.best_iteration})")

    # Average each fold's predictions across all folds
    test_probs += model.predict_proba(X_test) / n_splits

print(f"\nMean unseen-user accuracy: {np.mean(fold_accs):.4f} "
      f"(+/- {np.std(fold_accs):.4f})")

## 5. Generate predictions

Take the averaged class probabilities across folds, convert the highest-probability class
back to its activity label, and write the submission file. The class distribution is
printed as a sanity check that predictions are spread sensibly across activities rather
than collapsing onto one or two classes.


In [ ]:
final_preds = le.inverse_transform(test_probs.argmax(axis=1))

submission = pd.DataFrame({
    'user_snippet': meta_kaggle['user_snippet'],
    'prediction':   final_preds,
})
submission.to_csv('predictions_AK-67.csv', index=False)
print("Submission written to predictions_AK-67.csv")

print("\nPredicted class distribution (%):")
print((submission['prediction'].value_counts(normalize=True) * 100).round(2))